<a href="https://colab.research.google.com/github/iiilzxn/colab-work4harness/blob/main/nimimum_harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Environment
from pathlib import Path

WORKSPACE = Path("./workspace")
WORKSPACE.mkdir(exist_ok=True)

# 清理旧结果
result_file = WORKSPACE / "result.txt"
if result_file.exists():
    result_file.unlink()

# 创建输入文件
data_file = WORKSPACE / "data.txt"

data_file.write_text(
    "10\n20\n35\n17",
    encoding="utf-8"
)

print(data_file.read_text())

10
20
35
17


In [3]:
# Environment中的函数
def read_file(path):
    file_path = WORKSPACE / path
    return file_path.read_text(encoding="utf-8")


def write_file(path, content):
    file_path = WORKSPACE / path
    file_path.write_text(content, encoding="utf-8")

    return f"成功写入 {path}"

In [4]:
# 初始State
state = {
    "task": "读取 data.txt 中的数字，计算总和，并写入 result.txt",

    "step": 0,

    "status": "running",

    "last_observation": None,

    "observations": [],

    "artifacts": []
}

In [5]:
# Context Manager
def build_context(state):

    return {
        "task": state["task"],
        "step": state["step"],
        "last_observation": state["last_observation"],
        "artifacts": state["artifacts"]
    }

In [6]:
# 模拟LLM，固定read->write->finish
def fake_llm(context):

    step = context["step"]

    # 第一轮：读取 data.txt
    if step == 0:

        return {
            "name": "read_file",
            "arguments": {
                "path": "data.txt"
            }
        }

    # 第二轮：根据上一轮 Observation 计算结果
    elif step == 1:

        observation = context["last_observation"]

        content = observation["content"]

        numbers = [
            int(x)
            for x in content.splitlines()
        ]

        total = sum(numbers)

        return {
            "name": "write_file",
            "arguments": {
                "path": "result.txt",
                "content": str(total)
            }
        }

    # 后续认为任务结束
    else:

        return {
            "name": "finish",
            "arguments": {}
        }

In [7]:
# Action Interface，模型输出Action之后调用环境中的函数
TOOL_REGISTRY = {
    "read_file": read_file,
    "write_file": write_file
}


def execute_action(action):

    name = action["name"]
    arguments = action["arguments"]

    if name not in TOOL_REGISTRY:
        raise ValueError(f"未知 Action: {name}")

    tool = TOOL_REGISTRY[name]

    return tool(**arguments)

In [8]:
# Observation Interface——把外部环境转换成Harness可以理解的格式
def build_observation(action, raw_result, success=True):

    return {
        "action": action["name"],
        "success": success,
        "content": raw_result
    }

In [9]:
# 更新State
def update_state(state, action, observation):

    state["last_observation"] = observation

    state["observations"].append(
        observation
    )

    if (
        observation["success"]
        and action["name"] == "write_file"
    ):
        path = action["arguments"]["path"]

        if path not in state["artifacts"]:
            state["artifacts"].append(path)

    state["step"] += 1

In [10]:
# 验证
def verify(state):

    result_file = WORKSPACE / "result.txt"

    if not result_file.exists():
        return False

    content = result_file.read_text(
        encoding="utf-8"
    ).strip()

    return content == "82"

In [11]:
# Control Loop
def run_fake_agent(state, max_steps=10):

    while state["status"] == "running":

        print("\n" + "=" * 50)
        print(f"Step {state['step']}")
        print("=" * 50)

        # 防止死循环
        if state["step"] >= max_steps:
            state["status"] = "failed"
            print("超过最大执行轮数")
            break

        # 1. Context Manager
        context = build_context(state)

        print("\n[Context]")
        print(context)

        # 2. Fake LLM
        action = fake_llm(context)

        print("\n[Action]")
        print(action)

        # 模型声明结束
        if action["name"] == "finish":

            if verify(state):
                state["status"] = "completed"
                print("\n任务完成")
            else:
                state["status"] = "failed"
                print("\n模型想结束，但 Verify 不通过")

            break

        # 3. Action Interface
        try:
            raw_result = execute_action(action)
            success = True

        except Exception as e:
            raw_result = str(e)
            success = False

        print("\n[Environment Raw Result]")
        print(raw_result)

        # 4. Observation Interface
        observation = build_observation(
            action,
            raw_result,
            success
        )

        print("\n[Observation]")
        print(observation)

        # 5. Update State
        update_state(
            state,
            action,
            observation
        )

        print("\n[State]")
        print(state)

        # 6. Verify
        verified = verify(state)

        print("\n[Verify]")
        print(verified)

        if verified:
            state["status"] = "completed"
            print("\n任务验证成功")
            break

    return state

In [12]:
# 执行
final_state = run_fake_agent(state)


Step 0

[Context]
{'task': '读取 data.txt 中的数字，计算总和，并写入 result.txt', 'step': 0, 'last_observation': None, 'artifacts': []}

[Action]
{'name': 'read_file', 'arguments': {'path': 'data.txt'}}

[Environment Raw Result]
10
20
35
17

[Observation]
{'action': 'read_file', 'success': True, 'content': '10\n20\n35\n17'}

[State]
{'task': '读取 data.txt 中的数字，计算总和，并写入 result.txt', 'step': 1, 'status': 'running', 'last_observation': {'action': 'read_file', 'success': True, 'content': '10\n20\n35\n17'}, 'observations': [{'action': 'read_file', 'success': True, 'content': '10\n20\n35\n17'}], 'artifacts': []}

[Verify]
False

Step 1

[Context]
{'task': '读取 data.txt 中的数字，计算总和，并写入 result.txt', 'step': 1, 'last_observation': {'action': 'read_file', 'success': True, 'content': '10\n20\n35\n17'}, 'artifacts': []}

[Action]
{'name': 'write_file', 'arguments': {'path': 'result.txt', 'content': '82'}}

[Environment Raw Result]
成功写入 result.txt

[Observation]
{'action': 'write_file', 'success': True, 'content': '